In [1]:
import sys
print(sys.executable)

/Users/priyanshujha/Documents/SyncCare/.venv/bin/python


In [2]:
import pandas
import numpy
import sklearn
print("Environment Ready")

Environment Ready


In [6]:
import os
print(os.getcwd())

/Users/priyanshujha/Documents/SyncCare/ml/notebooks


In [8]:
import pandas as pd

DATASET_PATH = "../data/raw/dataset.csv"

nlice = pd.read_csv(
    DATASET_PATH,
    usecols=["PATIENT", "AGE", "GENDER", "SYMPTOMS", "PATHOLOGY"],
    dtype={
        "PATIENT": "string",
        "AGE": "int16",
        "GENDER": "string",
        "SYMPTOMS": "string",
        "PATHOLOGY": "string"
    }
)

print(nlice.shape)
nlice.head()

(3692607, 5)


,PATIENT,AGE,GENDER,SYMPTOMS,PATHOLOGY
0,b0915fdc-11f2-430a-aece-4f19f2d03b0f,40,M,Congestion::Head:Ear::::::39;Discharge::Head:E...,Otitis externa
1,bc8b5a00-f5cd-4675-bd26-68ff53101121,58,F,Congestion::Head:Ear::::::41;Discharge::Head:E...,Otitis externa
2,611a810d-eed5-4156-9d98-e375b3754df0,22,M,Inflammation::Head:Ear::::::27;Congestion::Hea...,Otitis externa
3,41542e6c-0b9c-464e-bb74-00f52b511f07,59,M,Congestion::Head:Ear::::::42;Discharge::Head:E...,Otitis externa
4,280423b0-3df5-416e-b446-0208679936bd,18,F,Congestion::Head:Ear::::::48;Discharge::Head:E...,Otitis externa


In [9]:
nlice.info()
print(nlice.isna().sum())
print("Unique patients:", nlice["PATIENT"].nunique())
print("Unique diseases:", nlice["PATHOLOGY"].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 3692607 entries, 0 to 3692606
Data columns (total 5 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   PATIENT    string
 1   AGE        int16 
 2   GENDER     string
 3   SYMPTOMS   string
 4   PATHOLOGY  string
dtypes: int16(1), string(4)
memory usage: 119.7 MB
PATIENT      0
AGE          0
GENDER       0
SYMPTOMS     0
PATHOLOGY    0
dtype: int64
Unique patients: 550363
Unique diseases: 55


In [11]:
def extract_symptom_location(symptom_entry):
    parts = symptom_entry.split(":")

    # Remove final numeric value
    parts = parts[:-1]

    if len(parts) < 8:
        return "", ""

    symptom = parts[0].strip()

    # Location occupies everything between:
    # symptom/nature and the final 5 fields
    location_parts = parts[2:-5]

    location = ":".join(
        part.strip()
        for part in location_parts
        if part.strip()
    )

    return symptom, location

In [12]:
def extract_record_features(symptom_string):
    symptoms = []
    locations = []

    for entry in str(symptom_string).split(";"):
        entry = entry.strip()

        if not entry:
            continue

        symptom, location = extract_symptom_location(entry)

        if symptom:
            symptoms.append(symptom)

        if location:
            locations.append(location)

    return symptoms, locations

In [13]:
parsed = nlice["SYMPTOMS"].map(extract_record_features)

nlice["SYMPTOMS_CLEAN"] = parsed.map(
    lambda x: "|".join(x[0])
)

nlice["LOCATION"] = parsed.map(
    lambda x: "|".join(x[1])
)

In [14]:
cleaned = nlice[
    [
        "PATIENT",
        "AGE",
        "GENDER",
        "SYMPTOMS_CLEAN",
        "LOCATION",
        "PATHOLOGY"
    ]
].copy()

cleaned.head()

,PATIENT,AGE,GENDER,SYMPTOMS_CLEAN,LOCATION,PATHOLOGY
0,b0915fdc-11f2-430a-aece-4f19f2d03b0f,40,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
1,bc8b5a00-f5cd-4675-bd26-68ff53101121,58,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
2,611a810d-eed5-4156-9d98-e375b3754df0,22,M,Inflammation|Congestion|Discharge|Altered_sens...,Head:Ear|Head:Ear|Head:Ear|Head:Ear,Otitis externa
3,41542e6c-0b9c-464e-bb74-00f52b511f07,59,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
4,280423b0-3df5-416e-b446-0208679936bd,18,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa


In [15]:
print("Shape:", cleaned.shape)

print("\nDiseases:")
print(cleaned["PATHOLOGY"].nunique())

print("\nUnique locations:")
print(
    cleaned["LOCATION"]
    .str.split("|")
    .explode()
    .nunique()
)

print("\nEmpty locations:")
print((cleaned["LOCATION"] == "").sum())

print("\nAverage symptoms per record:")
print(
    cleaned["SYMPTOMS_CLEAN"]
    .str.split("|")
    .str.len()
    .mean()
)

Shape: (3692607, 6)

Diseases:
55

Unique locations:
137

Empty locations:
457066

Average symptoms per record:
3.54642343471699


In [16]:
empty_location = cleaned[cleaned["LOCATION"] == ""]

print("Records with empty location:", len(empty_location))

display(
    empty_location[
        ["AGE", "GENDER", "SYMPTOMS_CLEAN", "PATHOLOGY"]
    ].head(20)
)

Records with empty location: 457066


,AGE,GENDER,SYMPTOMS_CLEAN,PATHOLOGY
280801,30,M,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
280808,30,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
280828,43,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281057,22,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281135,72,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281433,29,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281482,1,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281489,1,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281495,62,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency
281561,21,F,Dizziness|Seizure|Altered_mental_state,Vitamin B12 deficiency


In [17]:
empty_indices = empty_location.index[:20]

for idx in empty_indices:
    print(f"\n--- Row {idx} ---")
    print("Raw:")
    print(nlice.loc[idx, "SYMPTOMS"])

    print("Clean symptoms:")
    print(cleaned.loc[idx, "SYMPTOMS_CLEAN"])

    print("Location:")
    print(repr(cleaned.loc[idx, "LOCATION"]))


--- Row 280801 ---
Raw:
Dizziness:Light-headedness::::::::35;Seizure:::::::::41;Altered_mental_state:Loss_of_conciousness::::::::37
Clean symptoms:
Dizziness|Seizure|Altered_mental_state
Location:
''

--- Row 280808 ---
Raw:
Dizziness:Light-headedness::::::::43;Seizure:::::::::27;Altered_mental_state:Loss_of_conciousness::::::::40
Clean symptoms:
Dizziness|Seizure|Altered_mental_state
Location:
''

--- Row 280828 ---
Raw:
Dizziness:Light-headedness::::::::40;Seizure:::::::::30;Altered_mental_state:Loss_of_conciousness::::::::38
Clean symptoms:
Dizziness|Seizure|Altered_mental_state
Location:
''

--- Row 281057 ---
Raw:
Dizziness:Light-headedness::::::::33;Seizure:::::::::25;Altered_mental_state:Loss_of_conciousness::::::::31
Clean symptoms:
Dizziness|Seizure|Altered_mental_state
Location:
''

--- Row 281135 ---
Raw:
Dizziness:Light-headedness::::::::39;Seizure:::::::::49;Altered_mental_state:Loss_of_conciousness::::::::45
Clean symptoms:
Dizziness|Seizure|Altered_mental_state
Location

In [18]:
location_counts = (
    cleaned["LOCATION"]
    .str.split("|")
    .explode()
    .value_counts()
)

print("Unique locations:", location_counts.index.nunique())
display(location_counts.head(20))

Unique locations: 137


LOCATION
Chest                             461805
                                  457066
Joints                            449150
Head:Ear                          362016
Abdomen                           316121
Head                              235844
Lower_extremities                 217528
Extremities                       213600
Neck                              170845
Chest:Breast                      136321
Head:Eyes                         136270
Abdomen:Lower_abdomen             124357
Nose                              123456
Pelvis_area:Genital_area          114030
Pelvis_area:Urethra                78387
Pelvis_area:Vagina                 76916
Head:Post_aurical_(behind_ear)     72891
Chest:Nipple                       70735
Upper_extremities:Hand             69023
Hip:Peri-anal-inner_thighs         68837
Name: count, dtype: int64

In [19]:
OUTPUT_PATH = "../data/processed/nliceCleaned.csv"

cleaned.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {cleaned.shape}")

Saved: ../data/processed/nliceCleaned.csv
Shape: (3692607, 6)


In [20]:
test = pd.read_csv(OUTPUT_PATH)

print(test.shape)
display(test.head())

(3692607, 6)


,PATIENT,AGE,GENDER,SYMPTOMS_CLEAN,LOCATION,PATHOLOGY
0,b0915fdc-11f2-430a-aece-4f19f2d03b0f,40,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
1,bc8b5a00-f5cd-4675-bd26-68ff53101121,58,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
2,611a810d-eed5-4156-9d98-e375b3754df0,22,M,Inflammation|Congestion|Discharge|Altered_sens...,Head:Ear|Head:Ear|Head:Ear|Head:Ear,Otitis externa
3,41542e6c-0b9c-464e-bb74-00f52b511f07,59,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
4,280423b0-3df5-416e-b446-0208679936bd,18,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa


In [21]:
print("Shape:", cleaned.shape)

print("\nMissing values:")
print(cleaned.isna().sum())

print("\nUnique values:")
print("Patients:", cleaned["PATIENT"].nunique())
print("Diseases:", cleaned["PATHOLOGY"].nunique())
print("Gender:", cleaned["GENDER"].nunique())
print("Locations:", cleaned["LOCATION"].str.split("|").explode().nunique())
print("Symptoms:", cleaned["SYMPTOMS_CLEAN"].str.split("|").explode().nunique())

print("\nDisease distribution:")
print(cleaned["PATHOLOGY"].value_counts().describe())

print("\nSymptoms per record:")
print(cleaned["SYMPTOMS_CLEAN"].str.split("|").str.len().describe())

print("\nLocation entries per record:")
print(cleaned["LOCATION"].str.split("|").str.len().describe())

Shape: (3692607, 6)

Missing values:
PATIENT           0
AGE               0
GENDER            0
SYMPTOMS_CLEAN    0
LOCATION          0
PATHOLOGY         0
dtype: int64

Unique values:
Patients: 550363
Diseases: 55
Gender: 2
Locations: 137
Symptoms: 80

Disease distribution:
count            55.0
mean     67138.309091
std      10754.045289
min            7689.0
25%           68521.0
50%           68884.0
75%           69458.0
max           74217.0
Name: count, dtype: Float64

Symptoms per record:
count    3.692607e+06
mean     3.546423e+00
std      9.783847e-01
min      2.000000e+00
25%      3.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      1.100000e+01
Name: SYMPTOMS_CLEAN, dtype: float64

Location entries per record:
count    3.692607e+06
mean     1.604634e+00
std      8.682081e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      6.000000e+00
Name: LOCATION, dtype: float64
